Embedding using DistilBert send to train with LR, SVC, NB, and RF


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
data = pd.read_csv("training_data_lowercase.csv",sep='\t', names=['label', 'title'])
print(data.shape)
data.fillna("",inplace=True)
print(data.head())


(34152, 2)
   label                                              title
0      0  donald trump sends out embarrassing new year‚s...
1      0  drunk bragging trump staffer started russian c...
2      0  sheriff david clarke becomes an internet joke ...
3      0  trump is so obsessed he even has obama‚s name ...
4      0  pope francis just called out donald trump duri...


as part of pre proc we are able to see color codes in csv which are incorrectly interpretted by vs code
they are not color code but simply corresponds to episode num

we also see video/picture are there in some data points, while these are just metadata it could change the meaning of the sentence once we remove punctuations

we also see that this metadata in enclosed in () in Training sample while its enclosed in [] in testing data, so we might need different pre processing

In [5]:
from preProc import normalize_text

data["clean_text"] = data["title"].apply(normalize_text)
print(data.head)

<bound method NDFrame.head of        label                                              title  \
0          0  donald trump sends out embarrassing new year‚s...   
1          0  drunk bragging trump staffer started russian c...   
2          0  sheriff david clarke becomes an internet joke ...   
3          0  trump is so obsessed he even has obama‚s name ...   
4          0  pope francis just called out donald trump duri...   
...      ...                                                ...   
34147      1  tears in rain as thais gather for late king's ...   
34148      1  pyongyang university needs non-u.s. teachers a...   
34149      1  philippine president duterte to visit japan ah...   
34150      1  japan's abe may have won election\tbut many do...   
34151      1  demoralized and divided: inside catalonia's po...   

                                              clean_text  
0      donald trump sends out embarrassing new year s...  
1      drunk bragging trump staffer started rus

In [6]:
from sklearn.model_selection import train_test_split
X=data.drop(columns=['label'])
y=data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_val.shape}")
print(f"Training output size: {y_train.shape}")
print(f"Testing output size: {y_val.shape}")

Training set size: (27321, 2)
Testing set size: (6831, 2)
Training output size: (27321,)
Testing output size: (6831,)


Sentence Transformers Embedding

In [7]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

X_train_EMB_ST = embedding_model.encode(X_train["clean_text"].tolist(), show_progress_bar=True)
X_val_EMB_ST = embedding_model.encode(X_val["clean_text"].tolist(), show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/854 [00:00<?, ?it/s]

Batches:   0%|          | 0/214 [00:00<?, ?it/s]

DistilBERT Embeddings

In [9]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np

distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
distilbert.eval()

def get_distilbert_embeddings(texts, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')

        with torch.no_grad():
            outputs = distilbert(**inputs)

        embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
        all_embeddings.append(embeddings)

        if i % 500 == 0:
            print(f"Encoded {i}/{len(texts)}")

    return np.vstack(all_embeddings)

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

X_train_EMB = get_distilbert_embeddings(X_train["clean_text"].tolist())
X_val_EMB   = get_distilbert_embeddings(X_val["clean_text"].tolist())

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoded 0/27321
Encoded 4000/27321
Encoded 8000/27321
Encoded 12000/27321
Encoded 16000/27321
Encoded 20000/27321
Encoded 24000/27321
Encoded 0/6831
Encoded 4000/6831


In [10]:
from scipy.sparse import hstack, csr_matrix

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import numpy as np

experiments = {
    "DistilBERT"  : (csr_matrix(X_train_EMB), csr_matrix(X_val_EMB)),
    "Sentence Transformer": (csr_matrix(X_train_EMB_ST),csr_matrix(X_val_EMB_ST))
}

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC"         : LinearSVC(),
    "Random Forest"      : RandomForestClassifier(n_estimators=100)
}

results = []

for name, (Xtr, Xva) in experiments.items():
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        preds = model.predict(Xva)
        acc = accuracy_score(y_val, preds)
        results.append((name, model_name, acc))
        print(f"{name} : {model_name} : {acc:.4f}")


results_df = pd.DataFrame(results, columns=["Text Vectorization", "Model", "Validation Accuracy"]).sort_values(
    by="Validation Accuracy", ascending=False
)
print("\nValidation results:")
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)

DistilBERT : Logistic Regression : 0.9467
DistilBERT : Linear SVC : 0.9461
DistilBERT : Random Forest : 0.9122
Sentence Transformer : Logistic Regression : 0.9179
Sentence Transformer : Linear SVC : 0.9236
Sentence Transformer : Random Forest : 0.9009

Validation results:


,Text Vectorization,Model,Validation Accuracy
0,DistilBERT,Logistic Regression,0.946714
1,DistilBERT,Linear SVC,0.946128
4,Sentence Transformer,Linear SVC,0.923584
3,Sentence Transformer,Logistic Regression,0.917874
2,DistilBERT,Random Forest,0.912165
5,Sentence Transformer,Random Forest,0.900893


Best model: Logistic Regression
